# main


In [ ]:
import sys
sys.path.append('/path/to/project')

In [ ]:
from ultralytics import YOLO
import pandas as pd

from functions.pairs import form_pairs, pairs_to_dict, get_lwir_paths
from functions.single_model_predict import predict_images
from functions.ensemble import results_fusion_wr
from functions.helpers import filter_low_confidence, filter_results
from functions.evaluation import evaluate_results
from functions.empty_imgs_verification import verify_empty_rgb_with_lwir

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [ ]:
# Завантаження даних
rgb_test_path = '/content/rgb/test'
lwir_test_path = '/content/lwir/test'
rgb_images_path = rgb_test_path + '/images'
lwir_images_path = lwir_test_path + '/images'

In [ ]:
RGB_IMG_SIZE = (1216, 1936)
LWIR_IMG_SIZE = (512, 640)


CLASS_NAMES = ['other', 'PFM-1', 'PMN', 'M6', 'TMA-2', 'TC-3.6']

In [ ]:
# Створення пар
pairs_df = form_pairs(rgb_images_path, lwir_images_path, 'image_pairs.csv')
pair_dict = pairs_to_dict(pairs_df)

pairs count: 570


In [ ]:
model_rgb = YOLO("/content/models/models/yolov8n_epochs50_batch16_rgb.pt")
model_lwir = YOLO("/content/models/models/yolo11n_epochs50_batch16_lwir.pt")

# f1-score

In [ ]:
import optuna
import sys

def objective(trial):
    CONF_RGB = trial.suggest_categorical(
        'CONF_RGB',
        [0.001, 0.01, 0.1, 0.15]
    )
    THRESHOLD_RGB = trial.suggest_float('THRESHOLD_RGB', 0.1, 0.6, step=0.0001)
    DISTANCE_THRESHOLD = trial.suggest_float('DISTANCE_THRESHOLD', 0.1, 0.5, step=0.01)
    CONF_FOV_THRESHOLD = trial.suggest_float('CONF_FOV_THRESHOLD', 0.15, 0.6, step=0.0001)
    CONF_LWIR = trial.suggest_float('CONF_LWIR', 0.1, 0.9, step=0.0001)
    CONF_FOR_EMPTY = trial.suggest_float('CONF_FOR_EMPTY', 0.1, 0.9, step=0.0001)

    rgb_results_current = predict_images(
        model_rgb,
        rgb_images_path,
        None,
        conf_model=CONF_RGB
    )
    rgb_lowconfidence = filter_low_confidence(rgb_results_current, THRESHOLD_RGB)

    selected_lwir_paths = get_lwir_paths(rgb_lowconfidence, pair_dict, lwir_images_path)
    lwir_results = predict_images(model_lwir, selected_lwir_paths, None, conf_model=CONF_LWIR)

    notanobjectlist = results_fusion_wr(
        rgb_lowconfidence, lwir_results, pair_dict,
        RGB_IMG_SIZE, LWIR_IMG_SIZE,
        iou_threshold=IOU_THRESHOLD,
        threshold_rgb=THRESHOLD_RGB,
        distance_threshold=DISTANCE_THRESHOLD,
        conf_fov_threshold=CONF_FOV_THRESHOLD
    )

    filtered_rgb_results = filter_results(rgb_results_current, notanobjectlist)

    final_results = verify_empty_rgb_with_lwir(
        filtered_rgb_results, rgb_images_path, lwir_images_path,
        pair_dict, pairs_df, model_lwir,
        conf_for_empty=CONF_FOR_EMPTY
    )

    temp_results_file = f"temp_results_trial_{trial.number}.csv"
    df = pd.DataFrame(final_results)
    df.to_csv(temp_results_file, index=False)

    try:
        result = evaluate_results(
            temp_results_file,
            labels_dir='/content/rgb/test/labels',
            img_size=RGB_IMG_SIZE,
            iou_threshold=0.5,
            class_names=CLASS_NAMES
        )

        if isinstance(result, tuple):
            metrics, _ = result
            f1_score = metrics['overall']['f1']
        else:
            f1_score = result['overall']['f1']

    except Exception as e:
        print(f"Trial {trial.number} failed with error: {e}")
        import traceback
        traceback.print_exc()
        f1_score = 0.0

    import os
    if os.path.exists(temp_results_file):
        os.remove(temp_results_file)

    return f1_score

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

print("\n" + "="*30)
print("Optimization finished!")
print(f"Number of finished trials: {len(study.trials)}")
print("Best trial:")
trial = study.best_trial

print(f"  Value (F1-score): {trial.value}")
print("  Params: ")
for key, value in trial.params.items():
    print(f"    {key}: {value}")



[I 2025-11-07 11:28:46,836] A new study created in memory with name: no-name-c17f052a-68de-44bd-86b2-1fbcde1839fc


Empty RGB detections: 60


[I 2025-11-07 11:29:17,046] Trial 0 finished with value: 0.9341085271317829 and parameters: {'CONF_RGB': 0.01, 'THRESHOLD_RGB': 0.15940000000000001, 'DISTANCE_THRESHOLD': 0.30000000000000004, 'CONF_FOV_THRESHOLD': 0.315, 'CONF_LWIR': 0.1358, 'CONF_FOR_EMPTY': 0.3361}. Best is trial 0 with value: 0.9341085271317829.


Objects detected on LWIR: 0
Final total detections: 1076

metrics:
Precision: 0.8959
Recall:    0.9757
F1-Score:  0.9341
Accuracy:  0.8764

Counts:
True Positives:  964
False Positives: 112
False Negatives: 24

Per-class metrics:
Class: other
   Precision: 0.8418
   Recall: 0.9514
   TP: 431
   FN: 22
   FP: 81
Class: PFM-1
   Precision: 0.9401
   Recall: 0.9967
   TP: 298
   FN: 1
   FP: 19
Class: PMN
   Precision: 1.0000
   Recall: 1.0000
   TP: 90
   FN: 0
   FP: 0
Class: M6
   Precision: 0.9247
   Recall: 1.0000
   TP: 86
   FN: 0
   FP: 7
Class: TMA-2
   Precision: 0.9167
   Recall: 0.9778
   TP: 44
   FN: 1
   FP: 4
Class: TC-3.6
   Precision: 0.9375
   Recall: 1.0000
   TP: 15
   FN: 0
   FP: 1
Empty RGB detections: 61


[I 2025-11-07 11:29:33,752] Trial 1 finished with value: 0.9520514087988136 and parameters: {'CONF_RGB': 0.1, 'THRESHOLD_RGB': 0.2238, 'DISTANCE_THRESHOLD': 0.28, 'CONF_FOV_THRESHOLD': 0.49029999999999996, 'CONF_LWIR': 0.6315, 'CONF_FOR_EMPTY': 0.4596}. Best is trial 1 with value: 0.9520514087988136.


...

Objects detected on LWIR: 0
Final total detections: 1011

metrics:
Precision: 0.9535
Recall:    0.9757
F1-Score:  0.9645
Accuracy:  0.9314

Counts:
True Positives:  964
False Positives: 47
False Negatives: 24

Per-class metrics:
Class: other
   Precision: 0.9431
   Recall: 0.9514
   TP: 431
   FN: 22
   FP: 26
Class: PFM-1
   Precision: 0.9551
   Recall: 0.9967
   TP: 298
   FN: 1
   FP: 14
Class: PMN
   Precision: 1.0000
   Recall: 1.0000
   TP: 90
   FN: 0
   FP: 0
Class: M6
   Precision: 0.9451
   Recall: 1.0000
   TP: 86
   FN: 0
   FP: 5
Class: TMA-2
   Precision: 0.9565
   Recall: 0.9778
   TP: 44
   FN: 1
   FP: 2
Class: TC-3.6
   Precision: 1.0000
   Recall: 1.0000
   TP: 15
   FN: 0
   FP: 0
Empty RGB detections: 62


[I 2025-11-07 11:56:26,924] Trial 98 finished with value: 0.9639639639639639 and parameters: {'CONF_RGB': 0.1, 'THRESHOLD_RGB': 0.37840000000000007, 'DISTANCE_THRESHOLD': 0.39, 'CONF_FOV_THRESHOLD': 0.49640000000000006, 'CONF_LWIR': 0.5839, 'CONF_FOR_EMPTY': 0.521}. Best is trial 36 with value: 0.9644822411205604.


Objects detected on LWIR: 0
Final total detections: 1010

metrics:
Precision: 0.9535
Recall:    0.9747
F1-Score:  0.9640
Accuracy:  0.9304

Counts:
True Positives:  963
False Positives: 47
False Negatives: 25

Per-class metrics:
Class: other
   Precision: 0.9430
   Recall: 0.9492
   TP: 430
   FN: 23
   FP: 26
Class: PFM-1
   Precision: 0.9551
   Recall: 0.9967
   TP: 298
   FN: 1
   FP: 14
Class: PMN
   Precision: 1.0000
   Recall: 1.0000
   TP: 90
   FN: 0
   FP: 0
Class: M6
   Precision: 0.9451
   Recall: 1.0000
   TP: 86
   FN: 0
   FP: 5
Class: TMA-2
   Precision: 0.9565
   Recall: 0.9778
   TP: 44
   FN: 1
   FP: 2
Class: TC-3.6
   Precision: 1.0000
   Recall: 1.0000
   TP: 15
   FN: 0
   FP: 0
Empty RGB detections: 62


[I 2025-11-07 11:56:44,608] Trial 99 finished with value: 0.961 and parameters: {'CONF_RGB': 0.01, 'THRESHOLD_RGB': 0.4101, 'DISTANCE_THRESHOLD': 0.42000000000000004, 'CONF_FOV_THRESHOLD': 0.5178, 'CONF_LWIR': 0.5551, 'CONF_FOR_EMPTY': 0.494}. Best is trial 36 with value: 0.9644822411205604.


Objects detected on LWIR: 0
Final total detections: 1012

metrics:
Precision: 0.9496
Recall:    0.9727
F1-Score:  0.9610
Accuracy:  0.9249

Counts:
True Positives:  961
False Positives: 51
False Negatives: 27

Per-class metrics:
Class: other
   Precision: 0.9449
   Recall: 0.9470
   TP: 429
   FN: 24
   FP: 25
Class: PFM-1
   Precision: 0.9401
   Recall: 0.9967
   TP: 298
   FN: 1
   FP: 19
Class: PMN
   Precision: 1.0000
   Recall: 1.0000
   TP: 90
   FN: 0
   FP: 0
Class: M6
   Precision: 0.9444
   Recall: 0.9884
   TP: 85
   FN: 1
   FP: 5
Class: TMA-2
   Precision: 0.9565
   Recall: 0.9778
   TP: 44
   FN: 1
   FP: 2
Class: TC-3.6
   Precision: 1.0000
   Recall: 1.0000
   TP: 15
   FN: 0
   FP: 0

Optimization finished!
Number of finished trials: 100
Best trial:
  Value (F1-score): 0.9644822411205604
  Params: 
    CONF_RGB: 0.1
    THRESHOLD_RGB: 0.3781
    DISTANCE_THRESHOLD: 0.38
    CONF_FOV_THRESHOLD: 0.528
    CONF_LWIR: 0.5557000000000001
    CONF_FOR_EMPTY: 0.44290000000000

# f1-score and recall

In [ ]:
import optuna
import sys

def objective(trial):
    CONF_RGB = trial.suggest_categorical(
        'CONF_RGB',
        [0.001, 0.01, 0.1]
    )
    THRESHOLD_RGB = trial.suggest_float('THRESHOLD_RGB', 0.1, 0.6, step=0.0001)
    DISTANCE_THRESHOLD = trial.suggest_float('DISTANCE_THRESHOLD', 0.1, 0.5, step=0.01)
    CONF_FOV_THRESHOLD = trial.suggest_float('CONF_FOV_THRESHOLD', 0.15, 0.6, step=0.0001)
    CONF_LWIR = trial.suggest_float('CONF_LWIR', 0.1, 0.9, step=0.0001)
    CONF_FOR_EMPTY = trial.suggest_float('CONF_FOR_EMPTY', 0.1, 0.9, step=0.0001)

    rgb_results_current = predict_images(
        model_rgb,
        rgb_images_path,
        None,
        conf_model=CONF_RGB
    )
    rgb_lowconfidence = filter_low_confidence(rgb_results_current, THRESHOLD_RGB)

    selected_lwir_paths = get_lwir_paths(rgb_lowconfidence, pair_dict, lwir_images_path)
    lwir_results = predict_images(model_lwir, selected_lwir_paths, None, conf_model=CONF_LWIR)

    notanobjectlist = results_fusion_wr(
        rgb_lowconfidence, lwir_results, pair_dict,
        RGB_IMG_SIZE, LWIR_IMG_SIZE,
        iou_threshold=IOU_THRESHOLD,
        threshold_rgb=THRESHOLD_RGB,
        distance_threshold=DISTANCE_THRESHOLD,
        conf_fov_threshold=CONF_FOV_THRESHOLD
    )

    filtered_rgb_results = filter_results(rgb_results_current, notanobjectlist)

    final_results = verify_empty_rgb_with_lwir(
        filtered_rgb_results, rgb_images_path, lwir_images_path,
        pair_dict, pairs_df, model_lwir,
        conf_for_empty=CONF_FOR_EMPTY
    )

    temp_results_file = f"temp_results_trial_{trial.number}.csv"
    df = pd.DataFrame(final_results)
    df.to_csv(temp_results_file, index=False)

    try:
        result = evaluate_results(
            temp_results_file,
            labels_dir='/content/rgb/test/labels',
            img_size=RGB_IMG_SIZE,
            iou_threshold=0.5,
            class_names=CLASS_NAMES
        )

        if isinstance(result, tuple):
            metrics, _ = result
        else:
            metrics = result

        recall_score = metrics['overall'].get('recall', 0.0)
        f1_score = metrics['overall'].get('f1', 0.0)

        if recall_score is None: recall_score = 0.0
        if f1_score is None: f1_score = 0.0


    except Exception as e:
        print(f"Trial {trial.number} failed with error: {e}")
        import traceback
        traceback.print_exc()

        recall_score = 0.0
        f1_score = 0.0

    return recall_score, f1_score

In [ ]:
study = optuna.create_study(directions=['maximize', 'maximize'])

study.optimize(objective, n_trials=150)

print("\n" + "="*30)
print("Optimization finished!")
print(f"Number of finished trials: {len(study.trials)}")

print(f"\nFound {len(study.best_trials)} optimal trials:")

sorted_trials = sorted(study.best_trials, key=lambda t: t.values[0], reverse=True)

print("\n--- Top 5 Trials (sorted by Recall) ---")
for trial in sorted_trials[:5]:
    print(f"  Trial {trial.number}:")
    print(f"    Recall: {trial.values[0]:.4f}")
    print(f"    F1-score: {trial.values[1]:.4f}")
    print(f"    Params: {trial.params}")

best_overall_trial = sorted_trials[0]
print("\n--- Best Trial (Max Recall) ---")
print(f"Value (Recall, F1): {best_overall_trial.values}")
print("Params: ")
for key, value in best_overall_trial.params.items():
    print(f"    {key}: {value}")

[I 2025-11-07 12:07:44,012] A new study created in memory with name: no-name-dfb0bc22-bf00-41f8-a9aa-80ef4b5399cf


Empty RGB detections: 63


[I 2025-11-07 12:07:58,693] Trial 0 finished with values: [0.9736842105263158, 0.9576903932304629] and parameters: {'CONF_RGB': 0.1, 'THRESHOLD_RGB': 0.4721000000000001, 'DISTANCE_THRESHOLD': 0.31, 'CONF_FOV_THRESHOLD': 0.201, 'CONF_LWIR': 0.6064, 'CONF_FOR_EMPTY': 0.3134}.


Objects detected on LWIR: 1
Final total detections: 1021

metrics:
Precision: 0.9422
Recall:    0.9737
F1-Score:  0.9577
Accuracy:  0.9188

Counts:
True Positives:  962
False Positives: 59
False Negatives: 26

Per-class metrics:
Class: other
   Precision: 0.9286
   Recall: 0.9470
   TP: 429
   FN: 24
   FP: 33
Class: PFM-1
   Precision: 0.9521
   Recall: 0.9967
   TP: 298
   FN: 1
   FP: 15
Class: PMN
   Precision: 1.0000
   Recall: 1.0000
   TP: 90
   FN: 0
   FP: 0
Class: M6
   Precision: 0.9348
   Recall: 1.0000
   TP: 86
   FN: 0
   FP: 6
Class: TMA-2
   Precision: 0.9167
   Recall: 0.9778
   TP: 44
   FN: 1
   FP: 4
Class: TC-3.6
   Precision: 0.9375
   Recall: 1.0000
   TP: 15
   FN: 0
   FP: 1
Empty RGB detections: 65


[I 2025-11-07 12:08:15,407] Trial 1 finished with values: [0.9686234817813765, 0.9584376564847272] and parameters: {'CONF_RGB': 0.01, 'THRESHOLD_RGB': 0.4485, 'DISTANCE_THRESHOLD': 0.39, 'CONF_FOV_THRESHOLD': 0.3196, 'CONF_LWIR': 0.6785, 'CONF_FOR_EMPTY': 0.6295}.


...

Objects detected on LWIR: 0
Final total detections: 1014

metrics:
Precision: 0.9428
Recall:    0.9676
F1-Score:  0.9550
Accuracy:  0.9140

Counts:
True Positives:  956
False Positives: 58
False Negatives: 32

Per-class metrics:
Class: other
   Precision: 0.9216
   Recall: 0.9338
   TP: 423
   FN: 30
   FP: 36
Class: PFM-1
   Precision: 0.9551
   Recall: 0.9967
   TP: 298
   FN: 1
   FP: 14
Class: PMN
   Precision: 1.0000
   Recall: 1.0000
   TP: 90
   FN: 0
   FP: 0
Class: M6
   Precision: 0.9348
   Recall: 1.0000
   TP: 86
   FN: 0
   FP: 6
Class: TMA-2
   Precision: 0.9565
   Recall: 0.9778
   TP: 44
   FN: 1
   FP: 2
Class: TC-3.6
   Precision: 1.0000
   Recall: 1.0000
   TP: 15
   FN: 0
   FP: 0
Empty RGB detections: 61


[I 2025-11-07 12:47:17,722] Trial 148 finished with values: [0.9757085020242915, 0.9511593487913171] and parameters: {'CONF_RGB': 0.1, 'THRESHOLD_RGB': 0.2111, 'DISTANCE_THRESHOLD': 0.44000000000000006, 'CONF_FOV_THRESHOLD': 0.5265, 'CONF_LWIR': 0.6214, 'CONF_FOR_EMPTY': 0.4655}.


Objects detected on LWIR: 0
Final total detections: 1039

metrics:
Precision: 0.9278
Recall:    0.9757
F1-Score:  0.9512
Accuracy:  0.9069

Counts:
True Positives:  964
False Positives: 75
False Negatives: 24

Per-class metrics:
Class: other
   Precision: 0.8960
   Recall: 0.9514
   TP: 431
   FN: 22
   FP: 50
Class: PFM-1
   Precision: 0.9551
   Recall: 0.9967
   TP: 298
   FN: 1
   FP: 14
Class: PMN
   Precision: 1.0000
   Recall: 1.0000
   TP: 90
   FN: 0
   FP: 0
Class: M6
   Precision: 0.9348
   Recall: 1.0000
   TP: 86
   FN: 0
   FP: 6
Class: TMA-2
   Precision: 0.9167
   Recall: 0.9778
   TP: 44
   FN: 1
   FP: 4
Class: TC-3.6
   Precision: 0.9375
   Recall: 1.0000
   TP: 15
   FN: 0
   FP: 1
Empty RGB detections: 62


[I 2025-11-07 12:47:33,514] Trial 149 finished with values: [0.9726720647773279, 0.9477317554240632] and parameters: {'CONF_RGB': 0.01, 'THRESHOLD_RGB': 0.40470000000000006, 'DISTANCE_THRESHOLD': 0.31, 'CONF_FOV_THRESHOLD': 0.48250000000000004, 'CONF_LWIR': 0.37160000000000004, 'CONF_FOR_EMPTY': 0.47020000000000006}.


Objects detected on LWIR: 0
Final total detections: 1040

metrics:
Precision: 0.9240
Recall:    0.9727
F1-Score:  0.9477
Accuracy:  0.9007

Counts:
True Positives:  961
False Positives: 79
False Negatives: 27

Per-class metrics:
Class: other
   Precision: 0.8900
   Recall: 0.9470
   TP: 429
   FN: 24
   FP: 53
Class: PFM-1
   Precision: 0.9401
   Recall: 0.9967
   TP: 298
   FN: 1
   FP: 19
Class: PMN
   Precision: 1.0000
   Recall: 1.0000
   TP: 90
   FN: 0
   FP: 0
Class: M6
   Precision: 0.9444
   Recall: 0.9884
   TP: 85
   FN: 1
   FP: 5
Class: TMA-2
   Precision: 0.9565
   Recall: 0.9778
   TP: 44
   FN: 1
   FP: 2
Class: TC-3.6
   Precision: 1.0000
   Recall: 1.0000
   TP: 15
   FN: 0
   FP: 0

Optimization finished!
Number of finished trials: 150

Found 2 optimal trials (Pareto front):

--- Top 5 Trials (sorted by Recall) ---
  Trial 50:
    Recall: 0.9757
    F1-score: 0.9645
    Params: {'CONF_RGB': 0.1, 'THRESHOLD_RGB': 0.3758, 'DISTANCE_THRESHOLD': 0.36, 'CONF_FOV_THRESHOLD